# Insurance Claims Fraud Detection - Baseline Models
This notebook implements the baseline pipeline using Logistic Regression on structured features and TF-IDF + Logistic Regression on the synthesized claim narratives.
It uses manual oversampling for the TF-IDF pipeline to account for the class imbalance.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, average_precision_score, confusion_matrix
import json
import time
import os

OUTPUT_DIR = 'model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# 1. Load Dataset
# Using the fixed dataset that correctly correlates text signals to the fraud labels.
df = pd.read_csv('dataset1_gold_with_notes_wild.csv')
y = (df['claim_status'] == 'D').astype(int)

In [3]:
# 2. Define Features and Stratified Split
numeric_features = ['premium_amount', 'claim_amount', 'age', 'tenure', 'no_of_family_members',
                    'any_injury', 'police_report_available', 'incident_hour_of_the_day']
categorical_features = ['insurance_type', 'marital_status', 'employment_status',
                        'risk_segmentation', 'house_type', 'social_class',
                        'customer_education_level', 'incident_severity', 'authority_contacted']
text_feature = 'adjuster_notes'

X_struct = df[numeric_features + categorical_features]
X_text = df[text_feature].fillna('')

X_struct_train, X_struct_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_struct, X_text, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# 3. Structured Baseline (Logistic Regression)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)])

lr_struct = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

lr_struct.fit(X_struct_train, y_train)
start_time = time.time()
y_pred_struct = lr_struct.predict(X_struct_test)
y_prob_struct = lr_struct.predict_proba(X_struct_test)[:, 1]
latency_struct = time.time() - start_time

In [5]:
# 4. Text Baseline (TF-IDF + Explicit Oversampling + Logistic Regression)
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

# Manual oversampling array concatenation
fraud_idx = np.where(y_train == 1)[0]
legit_idx = np.where(y_train == 0)[0]
upsampled_fraud_idx = np.random.choice(fraud_idx, size=len(legit_idx), replace=True)
balanced_idx = np.concatenate([legit_idx, upsampled_fraud_idx])

X_train_bal = X_train_tfidf[balanced_idx]
y_train_bal = y_train.values[balanced_idx]

lr_text = LogisticRegression(max_iter=1000, random_state=42)
lr_text.fit(X_train_bal, y_train_bal)
start_time = time.time()
y_pred_text = lr_text.predict(X_test_tfidf)
y_prob_text = lr_text.predict_proba(X_test_tfidf)[:, 1]
latency_text = time.time() - start_time

In [6]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, average_precision_score, confusion_matrix
import numpy as np
import json
import os

def get_top_risk_words(tfidf_vectorizer, lr_model, top_n=10):
    """
    Extracts the words with the highest positive coefficients from the
    Logistic Regression model, indicating the strongest fraud signals.
    """
    feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
    # LogisticRegression coef_ array shape is (1, n_features) for binary classification
    coefs = lr_model.coef_[0]

    # Sort coefficients in descending order to get the most positive ones
    top_positive_indices = np.argsort(coefs)[::-1][:top_n]

    # Return as a dictionary of word -> coefficient score
    return {feature_names[i]: float(coefs[i]) for i in top_positive_indices}

def evaluate_model(y_true, y_pred, y_prob, latency):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_score': float(f1_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'inference_latency_sec': float(latency)
    }

# 1. Calculate Standard Metrics
metrics = {
    'structured_baseline': evaluate_model(y_test, y_pred_struct, y_prob_struct, latency_struct),
    'text_baseline': evaluate_model(y_test, y_pred_text, y_prob_text, latency_text)
}

# 2. Append Top Risk Words to the Text Baseline Metrics
metrics['text_baseline']['top_risk_words'] = get_top_risk_words(tfidf, lr_text, top_n=10)

# 3. Export to JSON
with open(os.path.join(OUTPUT_DIR, 'baseline_metrics_updated.json'), 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"Updated Metrics exported with Top Risk Words and Latency to {OUTPUT_DIR}/baseline_metrics_updated.json")

Updated Metrics exported with Top Risk Words and Latency to model_outputs/baseline_metrics_updated.json


In [7]:
import joblib
import os

# Export the structured data baseline model pipeline
joblib.dump(lr_struct, os.path.join(OUTPUT_DIR, 'structured_baseline_lr_model.pkl'))

# Export the text data baseline model pipeline
joblib.dump(lr_text, os.path.join(OUTPUT_DIR, 'text_baseline_tf_idf_lr_model.pkl'))

['model_outputs/text_baseline_tf_idf_lr_model.pkl']

In [8]:
from sklearn.model_selection import cross_val_predict
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Re-loading dataset to create isolated pipelines for Late Fusion
df_fusion = pd.read_csv('dataset1_gold_with_notes_wild.csv')
y_fusion = (df_fusion['claim_status'] == 'D').astype(int)
df_fusion[text_feature] = df_fusion[text_feature].fillna('')

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    df_fusion, y_fusion, test_size=0.2, random_state=42, stratify=y_fusion
)

# 1. Setup isolated preprocessors
preprocessor_struct_f = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

preprocessor_text_f = ColumnTransformer(transformers=[
    ('text', TfidfVectorizer(stop_words='english', max_features=1000), text_feature)
])

# Helper to extract Top Risk Words from Tree Ensembles
def get_top_tree_words(pipeline, top_n=10):
    tfidf = pipeline.named_steps['preprocessor'].named_transformers_['text']
    feature_names = tfidf.get_feature_names_out()
    importances = pipeline.named_steps['classifier'].feature_importances_
    top_idx = np.argsort(importances)[::-1][:top_n]
    return {feature_names[i]: float(importances[i]) for i in top_idx}

fusion_metrics = {}

In [9]:
# ==========================================
# 2. Random Forest Architecture
# ==========================================

# 2a. RF Structured Only
rf_struct = Pipeline(steps=[
    ('preprocessor', preprocessor_struct_f),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100))
])
rf_struct.fit(X_train_f, y_train_f)
t0 = time.time()
preds = rf_struct.predict(X_test_f)
probs = rf_struct.predict_proba(X_test_f)[:, 1]
lat = time.time() - t0
fusion_metrics['rf_structured'] = evaluate_model(y_test_f, preds, probs, lat)
joblib.dump(rf_struct, os.path.join(OUTPUT_DIR, 'rf_structured_model.pkl'))

# 2b. RF Text Only
rf_text = Pipeline(steps=[
    ('preprocessor', preprocessor_text_f),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100))
])
rf_text.fit(X_train_f, y_train_f)
t0 = time.time()
preds = rf_text.predict(X_test_f)
probs = rf_text.predict_proba(X_test_f)[:, 1]
lat = time.time() - t0
fusion_metrics['rf_text'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['rf_text']['top_risk_words'] = get_top_tree_words(rf_text)
joblib.dump(rf_text, os.path.join(OUTPUT_DIR, 'rf_text_model.pkl'))

# 2c. RF Fusion (Logistic Regression Meta-Classifier)
rf_meta_train_struct = cross_val_predict(rf_struct, X_train_f, y_train_f, cv=5, method='predict_proba')[:, 1]
rf_meta_train_text = cross_val_predict(rf_text, X_train_f, y_train_f, cv=5, method='predict_proba')[:, 1]
X_meta_train_rf = np.column_stack((rf_meta_train_struct, rf_meta_train_text))

lr_fusion_rf = LogisticRegression(class_weight='balanced', random_state=42)
lr_fusion_rf.fit(X_meta_train_rf, y_train_f)

t0 = time.time()
rf_meta_test_struct = rf_struct.predict_proba(X_test_f)[:, 1]
rf_meta_test_text = rf_text.predict_proba(X_test_f)[:, 1]
X_meta_test_rf = np.column_stack((rf_meta_test_struct, rf_meta_test_text))
preds = lr_fusion_rf.predict(X_meta_test_rf)
probs = lr_fusion_rf.predict_proba(X_meta_test_rf)[:, 1]
lat = time.time() - t0
fusion_metrics['rf_fusion_lr'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['rf_fusion_lr']['lr_coefficients'] = {"rf_struct_signal": float(lr_fusion_rf.coef_[0][0]), "rf_text_signal": float(lr_fusion_rf.coef_[0][1])}
joblib.dump(lr_fusion_rf, os.path.join(OUTPUT_DIR, 'rf_fusion_meta_model.pkl'))

['model_outputs/rf_fusion_meta_model.pkl']

In [10]:
# ==========================================
# 3. XGBoost Architecture
# ==========================================
scale_pos_weight = (len(y_train_f) - sum(y_train_f)) / sum(y_train_f)

# 3a. XGB Structured Only
xgb_struct = Pipeline(steps=[
    ('preprocessor', preprocessor_struct_f),
    ('classifier', xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'))
])
xgb_struct.fit(X_train_f, y_train_f)
t0 = time.time()
preds = xgb_struct.predict(X_test_f)
probs = xgb_struct.predict_proba(X_test_f)[:, 1]
lat = time.time() - t0
fusion_metrics['xgb_structured'] = evaluate_model(y_test_f, preds, probs, lat)
joblib.dump(xgb_struct, os.path.join(OUTPUT_DIR, 'xgb_structured_model.pkl'))

# 3b. XGB Text Only
xgb_text = Pipeline(steps=[
    ('preprocessor', preprocessor_text_f),
    ('classifier', xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'))
])
xgb_text.fit(X_train_f, y_train_f)
t0 = time.time()
preds = xgb_text.predict(X_test_f)
probs = xgb_text.predict_proba(X_test_f)[:, 1]
lat = time.time() - t0
fusion_metrics['xgb_text'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['xgb_text']['top_risk_words'] = get_top_tree_words(xgb_text)
joblib.dump(xgb_text, os.path.join(OUTPUT_DIR, 'xgb_text_model.pkl'))

# 3c. XGB Fusion (Logistic Regression Meta-Classifier)
xgb_meta_train_struct = cross_val_predict(xgb_struct, X_train_f, y_train_f, cv=5, method='predict_proba')[:, 1]
xgb_meta_train_text = cross_val_predict(xgb_text, X_train_f, y_train_f, cv=5, method='predict_proba')[:, 1]
X_meta_train_xgb = np.column_stack((xgb_meta_train_struct, xgb_meta_train_text))

lr_fusion_xgb = LogisticRegression(class_weight='balanced', random_state=42)
lr_fusion_xgb.fit(X_meta_train_xgb, y_train_f)

t0 = time.time()
xgb_meta_test_struct = xgb_struct.predict_proba(X_test_f)[:, 1]
xgb_meta_test_text = xgb_text.predict_proba(X_test_f)[:, 1]
X_meta_test_xgb = np.column_stack((xgb_meta_test_struct, xgb_meta_test_text))
preds = lr_fusion_xgb.predict(X_meta_test_xgb)
probs = lr_fusion_xgb.predict_proba(X_meta_test_xgb)[:, 1]
lat = time.time() - t0
fusion_metrics['xgb_fusion_lr'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['xgb_fusion_lr']['lr_coefficients'] = {"xgb_struct_signal": float(lr_fusion_xgb.coef_[0][0]), "xgb_text_signal": float(lr_fusion_xgb.coef_[0][1])}
joblib.dump(lr_fusion_xgb, os.path.join(OUTPUT_DIR, 'xgb_fusion_meta_model.pkl'))

# ==========================================
# 4. Final Export
# ==========================================
with open(os.path.join(OUTPUT_DIR, 'late_fusion_metrics.json'), 'w') as f:
    json.dump(fusion_metrics, f, indent=4)

print("Training complete. Exported 6 base models, 2 meta models, and full metrics payload.")

Training complete. Exported 6 base models, 2 meta models, and full metrics payload.


In [11]:
# Install required libraries if you haven't already:
# !pip install transformers torch

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import joblib
import json
import time
import os
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
import xgboost as xgb

# ==========================================
# 1. FinBERT Embedding Extraction
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
finbert = AutoModel.from_pretrained('ProsusAI/finbert').to(device)

def get_finbert_embeddings(texts, batch_size=32):
    finbert.eval()
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = finbert(**encoded)
            # Extract the [CLS] token representation (768-dim)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

print("Extracting FinBERT embeddings for training set...")
X_train_text_emb = get_finbert_embeddings(X_train_f[text_feature])
print("Extracting FinBERT embeddings for test set...")
t0_emb = time.time()
X_test_text_emb = get_finbert_embeddings(X_test_f[text_feature])
emb_latency = time.time() - t0_emb

# ==========================================
# 2. FinBERT on Text Only
# ==========================================
lr_finbert_text = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_finbert_text.fit(X_train_text_emb, y_train_f)

t0 = time.time()
y_pred_fb_text = lr_finbert_text.predict(X_test_text_emb)
y_prob_fb_text = lr_finbert_text.predict_proba(X_test_text_emb)[:, 1]
lat = time.time() - t0 + emb_latency

fusion_metrics['finbert_text'] = evaluate_model(y_test_f, y_pred_fb_text, y_prob_fb_text, lat)

# Transformer Risk Word Approximation
def extract_transformer_risk_words(texts, probabilities, top_n=10):
    sorted_indices = np.argsort(probabilities)[::-1]
    top_risk_texts = texts.iloc[sorted_indices[:100]].tolist()

    vec = TfidfVectorizer(stop_words='english', max_features=1000)
    vec.fit(texts)
    high_risk_tfidf = vec.transform(top_risk_texts).mean(axis=0).A1

    feature_names = vec.get_feature_names_out()
    top_indices = np.argsort(high_risk_tfidf)[::-1][:top_n]
    return {feature_names[i]: float(high_risk_tfidf[i]) for i in top_indices}

fusion_metrics['finbert_text']['top_risk_words'] = extract_transformer_risk_words(X_test_f[text_feature], y_prob_fb_text)
joblib.dump(lr_finbert_text, os.path.join(OUTPUT_DIR, 'finbert_text_head_model.pkl'))

# ==========================================
# 3. Early Fusion (Embeddings + Structured Features in single Tree)
# ==========================================
X_train_struct_proc = preprocessor_struct_f.fit_transform(X_train_f)

t0_struct_proc = time.time()
X_test_struct_proc = preprocessor_struct_f.transform(X_test_f)
struct_proc_latency = time.time() - t0_struct_proc

X_train_early_fusion = np.hstack((X_train_struct_proc, X_train_text_emb))
X_test_early_fusion = np.hstack((X_test_struct_proc, X_test_text_emb))

# RF Early Fusion
rf_early = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf_early.fit(X_train_early_fusion, y_train_f)
t0 = time.time()
preds = rf_early.predict(X_test_early_fusion)
probs = rf_early.predict_proba(X_test_early_fusion)[:, 1]
lat = time.time() - t0 + emb_latency + struct_proc_latency
fusion_metrics['rf_early_fusion_finbert'] = evaluate_model(y_test_f, preds, probs, lat)
joblib.dump(rf_early, os.path.join(OUTPUT_DIR, 'rf_early_fusion_finbert.pkl'))

# XGB Early Fusion
xgb_early = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb_early.fit(X_train_early_fusion, y_train_f)
t0 = time.time()
preds = xgb_early.predict(X_test_early_fusion)
probs = xgb_early.predict_proba(X_test_early_fusion)[:, 1]
lat = time.time() - t0 + emb_latency + struct_proc_latency
fusion_metrics['xgb_early_fusion_finbert'] = evaluate_model(y_test_f, preds, probs, lat)
joblib.dump(xgb_early, os.path.join(OUTPUT_DIR, 'xgb_early_fusion_finbert.pkl'))

# ==========================================
# 4. Late Fusion (Meta-Classifier combining FinBERT + Structured models)
# ==========================================
fb_meta_train = cross_val_predict(lr_finbert_text, X_train_text_emb, y_train_f, cv=5, method='predict_proba')[:, 1]

# RF Late Fusion
X_meta_train_rf_fb = np.column_stack((rf_meta_train_struct, fb_meta_train))
lr_fusion_rf_fb = LogisticRegression(class_weight='balanced', random_state=42)
lr_fusion_rf_fb.fit(X_meta_train_rf_fb, y_train_f)

t0 = time.time()
rf_meta_test_struct = rf_struct.predict_proba(X_test_f)[:, 1]
X_meta_test_rf_fb = np.column_stack((rf_meta_test_struct, y_prob_fb_text))
preds = lr_fusion_rf_fb.predict(X_meta_test_rf_fb)
probs = lr_fusion_rf_fb.predict_proba(X_meta_test_rf_fb)[:, 1]
lat = time.time() - t0 + emb_latency
fusion_metrics['rf_late_fusion_finbert'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['rf_late_fusion_finbert']['lr_coefficients'] = {
    "rf_struct_signal": float(lr_fusion_rf_fb.coef_[0][0]),
    "finbert_text_signal": float(lr_fusion_rf_fb.coef_[0][1])
}
joblib.dump(lr_fusion_rf_fb, os.path.join(OUTPUT_DIR, 'rf_late_fusion_finbert.pkl'))

# XGB Late Fusion
X_meta_train_xgb_fb = np.column_stack((xgb_meta_train_struct, fb_meta_train))
lr_fusion_xgb_fb = LogisticRegression(class_weight='balanced', random_state=42)
lr_fusion_xgb_fb.fit(X_meta_train_xgb_fb, y_train_f)

t0 = time.time()
xgb_meta_test_struct = xgb_struct.predict_proba(X_test_f)[:, 1]
X_meta_test_xgb_fb = np.column_stack((xgb_meta_test_struct, y_prob_fb_text))
preds = lr_fusion_xgb_fb.predict(X_meta_test_xgb_fb)
probs = lr_fusion_xgb_fb.predict_proba(X_meta_test_xgb_fb)[:, 1]
lat = time.time() - t0 + emb_latency
fusion_metrics['xgb_late_fusion_finbert'] = evaluate_model(y_test_f, preds, probs, lat)
fusion_metrics['xgb_late_fusion_finbert']['lr_coefficients'] = {
    "xgb_struct_signal": float(lr_fusion_xgb_fb.coef_[0][0]),
    "finbert_text_signal": float(lr_fusion_xgb_fb.coef_[0][1])
}
joblib.dump(lr_fusion_xgb_fb, os.path.join(OUTPUT_DIR, 'xgb_late_fusion_finbert.pkl'))

# ==========================================
# 5. Final Export
# ==========================================
with open(os.path.join(OUTPUT_DIR, 'transformer_fusion_metrics.json'), 'w') as f:
    json.dump(fusion_metrics, f, indent=4)
print(f"Transformer models trained. Metrics exported to '{OUTPUT_DIR}/transformer_fusion_metrics.json'")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting FinBERT embeddings for training set...
Extracting FinBERT embeddings for test set...
Transformer models trained. Metrics exported to 'model_outputs/transformer_fusion_metrics.json'


In [12]:
import shutil
from google.colab import files

# 1. Zip the directory (creates 'my_archive.zip')
# Parameters: (target_zip_name, format, folder_to_zip)
shutil.make_archive('model_outputs', 'zip', '/content/model_outputs')

# 2. Trigger the browser download automatically
files.download('model_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>